# 1、ChatMessageHistory的使用
场景1：记忆存储

In [4]:
import os
from http.client import responses

import dotenv
from IPython.core.debugger import prompt
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.summarize.refine_prompts import prompt_template

from langchain_openai import ChatOpenAI

# 1、ChatMessageHistory的实例化

from langchain_community.chat_message_histories import ChatMessageHistory

# 1、ChatMessageHistory实例

history = ChatMessageHistory()

# 2、添加相关的消息进行存储

history.add_user_message("你好")
history.add_ai_message("你好，很高兴认识你，我叫小智")

# 3、打印存储的消息

print(history.messages)

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，很高兴认识你，我叫小智', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


场景2：对接LLM

In [5]:

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("DEEPSEEK_BASE_URL")

llm = ChatOpenAI(model="deepseek-chat")


In [6]:

# 1、ChatMessageHistory的实例化

history = ChatMessageHistory()

# 2、添加相关的消息存储
history.add_user_message("你好")
history.add_ai_message("你好，很高兴认识你，我是小智")
history.add_user_message("请帮我计算1 + 2 * 3 = ?")

response = llm.invoke(history.messages)

print(response.content)



根据数学运算的优先级，乘法（`*`）优先于加法（`+`），所以：

1 + 2 * 3  
= 1 + (2 * 3)  
= 1 + 6  
= **7**

答案是 **7**。


# 2、ConversationBufferMemory的使用
举例1：返回存储的字符串信息

In [13]:
from langchain_classic.memory import ConversationBufferMemory

# 1、ConversationBufferMemory的实例化
memory = ConversationBufferMemory()

# 2、存储相关的消息
# inputs对应的就是用户消息，outputs对应的就是ai消息

memory.save_context(inputs={"input":"你好，我叫小明"},outputs={"output":"很高兴认识你"})
memory.save_context(inputs={"input":"帮我回答一下1+2*3=?"},outputs={"output":"7"})

# 3、获取存储的信息
print(memory.load_memory_variables({}))




{'history': [HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答一下1+2*3=?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


举例2：以消息列表的方式返回存储信息

In [14]:
from langchain_classic.memory import ConversationBufferMemory

# 1、ConversationBufferMemory的实例化
memory = ConversationBufferMemory(return_messages=True)

# 2、存储相关的消息
# inputs对应的就是用户消息，outputs对应的就是ai消息

memory.save_context(inputs={"input":"你好，我叫小明"},outputs={"output":"很高兴认识你"})
memory.save_context(inputs={"input":"帮我回答一下1+2*3=?"},outputs={"output":"7"})

# 3、获取存储的信息
print(memory.load_memory_variables({}))

# 说明：返回的字典结构的key叫history


{'history': [HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答一下1+2*3=?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


举例3：结合大模型、提示词模板的使用(PromptTemplate)

In [18]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import LLMChain
# 1、创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

    当前对话历史：{history}

    人类问题：{question}

    回复：
    """
)

# 3、提供memory实例
memory = ConversationBufferMemory()

# 4、提供Chain
chain = LLMChain(llm=llm,prompt=prompt_template,memory=memory)

response = chain.invoke({"question":"你好，我的名字叫小明"})
response = chain.invoke({"question":"what is my name "})
print(response)

{'question': 'what is my name ', 'history': 'Human: 你好，我的名字叫小明\nAI: 你好，小明！很高兴认识你。有什么我可以帮助你的吗？', 'text': '你的名字是小明。'}


举例4：基于举例3，显示的设置memory的key的值

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
# 1、创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

    当前对话历史：{chat_history}

    人类问题：{question}

    回复：
    """
)

# 3、提供memory实例
memory = ConversationBufferMemory(memory_key="chat_history")

# 4、提供Chain
chain = LLMChain(llm=llm,prompt=prompt_template,memory=memory)

response = chain.invoke({"question":"你好，我的名字叫小明"})
response = chain.invoke({"question":"what is my name "})
print(response)

举例5：结合大模型、提示词模板的使用（ChatPromptTempalte）

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.prompts.chat import MessagesPlaceholder
# 1、创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")

# 2、提供提示词模板
prompt_template = ChatPromptTemplate.from_messages(
   [
       ("system","你可以和人类对话"),
       MessagesPlaceholder(variable_name="history"),
       ("human","{question}")
   ]
)

# 3、提供memory实例
memory = ConversationBufferMemory(return_messages=True)

# 4、提供Chain
chain = LLMChain(llm=llm,prompt=prompt_template,memory=memory)

res1 = chain.invoke({"question":"你好，我的名字叫小明"})
print(res1,end="\n\n")


{'question': '你好，我的名字叫小明', 'history': [HumanMessage(content='你好，我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，小明！很高兴认识你。有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'text': '你好，小明！很高兴认识你。有什么我可以帮助你的吗？'}



In [23]:
res2 = chain.invoke({"question":"what is my last question"})
print(res2)

{'question': 'what is my last question', 'history': [HumanMessage(content='你好，我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，小明！很高兴认识你。有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is my last question', additional_kwargs={}, response_metadata={}), AIMessage(content='你刚才的问题是：“what is my last question”，意思是“我的上一个问题是什么”。  \n而在此之前，你最初说的是：“你好，我的名字叫小明”。  \n\n所以，你上一个问题就是询问你的上一个问题是什么。需要我帮你解答其他问题吗？😊', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'text': '你刚才的问题是：“what is my last question”，意思是“我的上一个问题是什么”。  \n而在此之前，你最初说的是：“你好，我的名字叫小明”。  \n\n所以，你上一个问题就是询问你的上一个问题是什么。需要我帮你解答其他问题吗？😊'}


# 3、ConversationChain的使用
举例1：以PromptTemplate为例

In [25]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import LLMChain
from langchain_classic.chains import ConversationChain
# 1、创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

    当前对话历史：{history}

    人类问题：{input}

    回复：
    """
)

# 3、创建ConversationChain的实例
chain = ConversationChain(llm=llm,prompt=prompt_template)

response = chain.invoke({"input":"你好，我的名字叫小明"})
# response = chain.invoke({"question":"what is my name "})
print(response)

{'input': '你好，我的名字叫小明', 'history': '', 'response': '你好，小明！很高兴认识你。有什么我可以帮助你的吗？'}


举例2：使用默认提供的提示词模板

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import LLMChain
from langchain_classic.chains import ConversationChain
# 1、创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")


# 3、创建ConversationChain的实例（内部提供了默认的提示此模板。而此模板中的变量是{input}、{history}）
chain = ConversationChain(llm=llm)

response = chain.invoke({"input":"你好，我的名字叫小明"})
# response = chain.invoke({"question":"what is my name "})
print(response)